# 금감원 바로 이 목소리 셀레니움 자동 다운로드 구현

# 위스퍼 설치

In [2]:
!pip install git+https://github.com/openai/whisper.git
!pip install torch torchvision torchaudio

  Cloning https://github.com/openai/whisper.git to c:\users\user\appdata\local\temp\pip-req-build-ar18vt2e
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 20.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -------------------------- ------------- 8.4/12.6 MB 41.3 MB/s eta 0:00:01
   ---------------------------- ----------- 8.9/12.6 MB 23.2 MB/s eta 0:00:01
   --------------------

  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git 'C:\Users\user\AppData\Local\Temp\pip-req-build-ar18vt2e'

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\user\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 12.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 30.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ------------------------------------ --- 6.3/7.0 MB 30.5 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 27.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\user\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [1]:
import sys

print(sys.executable)

c:\Users\user\Downloads\데이터 비즈니스 분석 프로젝트 자료\woogawooga_project\.venv\Scripts\python.exe


# 셀레니움으로 금감원 비디오자료 자동 다운로드 구현

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
import requests, os

DOWNLOAD_DIR = "videos"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)


# 드라이버 초기화 함수
def init_driver():
    service = Service(ChromeDriverManager().install())
    opts = webdriver.ChromeOptions()
    opts.add_argument("--headless")
    return webdriver.Chrome(service=service, options=opts)


driver = init_driver()

for page in range(1, 13):
    try:
        video_page_urls = []
        url = f"https://www.fss.or.kr/fss/bbs/B0000203/list.do?menuNo=200686&pageIndex={page}"
        driver.get(url)

        # 링크 수집
        elements = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div.bd-list-thumb-a a")
            )
        )
        for el in elements:
            video_page_urls.append(el.get_attribute("href"))

        for idx, post_url in enumerate(video_page_urls, 1):
            driver.get(post_url)
            title = (
                WebDriverWait(driver, 10)
                .until(EC.presence_of_element_located((By.CSS_SELECTOR, "span.name")))
                .text
            )
            video = driver.find_element(By.TAG_NAME, "video").get_attribute("src")
            filename = "".join(
                c for c in title if c.isalnum() or c in (" ", "_")
            ).replace(" ", "_")
            path = os.path.join(DOWNLOAD_DIR, f"{filename}.mp4")

            r = requests.get(video, stream=True, timeout=30)
            if r.status_code == 200:
                with open(path, "wb") as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
    except WebDriverException:
        driver.quit()
        driver = init_driver()
    except Exception as e:
        print(f"오류: {e}")

driver.quit()